In [ ]:
#Temporal interpolation 
import numpy as np
from scipy.interpolate import CubicSpline

input_file_path = r"Your Path\Support_files\16_bc_interpolated_cm_corrected.txt"
output_file_path = r"Your Path\Support_files\17_bc_complete_interpolation_correction.txt"

def temporal_interpolation(input_path, output_path, new_timesteps=201):
    with open(input_path, 'r') as file:
        lines = file.readlines()

    # Read the header
    header = lines[0].strip()
    num_points, old_timesteps = map(int, header.split())
    new_time_array = np.linspace(0, (old_timesteps - 1) * 0.98 / (old_timesteps - 1), new_timesteps)

    interpolated_data = [f"{num_points} {new_timesteps}\n"]  # Initialize new file content with updated header
    i = 1  # Start after the header line

    while i < len(lines):
        # Read spatial point line (x, y, z, temporal points, id)
        spatial_point_line = lines[i].strip().split()
        x, y, z = map(float, spatial_point_line[:3])
        point_id = int(spatial_point_line[-1])
        interpolated_data.append(f"{x:.6f} {y:.6f} {z:.6f} {new_timesteps} {point_id}\n")
        i += 1

        # Extract the velocity and time data for the current point
        velocities = []
        times = []
        for _ in range(old_timesteps):
            velocity_line = lines[i].strip().split()
            u, v, w, t = map(float, velocity_line)
            velocities.append([u, v, w])
            times.append(t)
            i += 1

        # Convert to numpy array for processing
        velocities = np.array(velocities)
        times = np.array(times)

        # Perform cubic spline interpolation for each velocity component
        interpolated_u = CubicSpline(times, velocities[:, 0])(new_time_array)
        interpolated_v = CubicSpline(times, velocities[:, 1])(new_time_array)
        interpolated_w = CubicSpline(times, velocities[:, 2])(new_time_array)

        # Write interpolated data
        for t_idx, t in enumerate(new_time_array):
            interpolated_data.append(
                f"{interpolated_u[t_idx]:.6f} {interpolated_v[t_idx]:.6f} {interpolated_w[t_idx]:.6f} {t:.6f}\n"
            )

    # Save the interpolated data to the output file
    with open(output_path, 'w') as output_file:
        output_file.writelines(interpolated_data)

    print(f"Interpolated data saved to: {output_path}")

# Call the function to process and interpolate the file
temporal_interpolation(input_file_path, output_file_path, new_timesteps=201)